### Synthetic FM dataset generator


*multi-station composite → AWGN injection → branch-wise IQ imbalance on signal and AWGN → DC offset added to the combined analog waveform → ADC drive scaling → pre-quantization utilization measurement → ADC quantization/clipping.*

The pipeline is designed to model cascaded analog impairments:

 -- IQ imbalance is applied before summing signal and noise branches, preserving physical realism.

 -- DC offset and ADC drive are applied after impairments, as in a real analog front-end.

 -- Post-quantization metrics (post_sindr_db) capture the total degradation from all sources.



The script creates a dataset of complex baseband IQ captures that simulate receiving multiple FM radio stations simultaneously, with realistic analog and digital impairments, culminating in 8-bit ADC quantization—mimicking the output of a low-cost SDR like HackRF.

The generation follows a structured pipeline:

1. Multi-Station FM Synthesis

Creates 2–4 independent FM stations at different frequency offsets within the 8 MHz bandwidth.

Each station's audio is synthesized as a sum of sinusoids (300 Hz – 15 kHz) plus colored noise, upsampled from 48 kHz to 8 MHz, and used to frequency-modulate a carrier.

2. AWGN Injection

Adds complex Gaussian noise at a random SNR (8–30 dB) to simulate receiver thermal noise.

3. IQ Imbalance (Widely-Linear Model)

Applies gain imbalance (±0.9 dB) and phase imbalance (±6°) separately to the signal and noise components, mimicking analog mixer imperfections.

4. DC Offset Addition

Adds a complex DC offset (0.02–0.12 amplitude) to model LO leakage or bias issues.

5. ADC Drive Scaling

Scales the entire waveform by a gain factor (0.35–1.35) to adjust how much of the ADC’s full-scale range is utilized.

6. ADC Utilization Measurement

Before quantization, measures peak, RMS, and percentiles of I/Q amplitudes to assess how well the ADC range is used.

7. 8-Bit Complex Quantization & Clipping

Simulates an 8-bit ADC: I and Q are clipped to ±1, then quantized to 8-bit signed integers, and scaled back to ±1.



In [1]:
import json
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from math import gcd


# -------------------------- CONFIG -------------------------- #
NUM_EXAMPLES = 3
SAMPLE_RATE = 8_000_000
NUM_IQ_SAMPLES = 524_288
SEED_BASE = 12345

CENTER_FREQ_HZ = 98_000_000

FM_DEVIATION_HZ_MIN = 55_000.0
FM_DEVIATION_HZ_MAX = 75_000.0
FM_AUDIO_MAX_HZ = 15_000.0

EDGE_GUARD_HZ = 300_000
CHANNEL_SPACING_HZ = 200_000
MIN_STATION_SEPARATION_HZ = 400_000

NOISE_SNR_DB_RANGE = (8.0, 30.0)
STATION_GAIN_DB_RANGE = (-10.0, -1.0)

# HackRF-like impairments
DC_OFFSET_RANGE = (0.02, 0.12)
IQ_GAIN_IMBALANCE_DB_RANGE = (-0.9, 0.9)
IQ_PHASE_IMBALANCE_DEG_RANGE = (-6.0, 6.0)
ADC_DRIVE_RANGE = (0.35, 1.35)

# Plotting
ENABLE_PLOTS = False
PSD_NPERSEG = 8192

# Output
OUT_DIR = Path(f"dataset_fm_hackrf_{NUM_EXAMPLES}")

# Time vector:
T_VECTOR = np.arange(NUM_IQ_SAMPLES, dtype=np.float64) / SAMPLE_RATE

# ----- audio-domain constants -----
AUDIO_SAMPLE_RATE = 48_000                          # Hz — proper audio rate
AUDIO_NUM_SAMPLES = int(NUM_IQ_SAMPLES * AUDIO_SAMPLE_RATE / SAMPLE_RATE)   # 3145 samples

# Upsample ratio as a reduced fraction
_g = gcd(SAMPLE_RATE, AUDIO_SAMPLE_RATE)
_UP   = SAMPLE_RATE      // _g          # 500
_DOWN = AUDIO_SAMPLE_RATE // _g         # 3

# Time vector for audio domain only
T_AUDIO = np.arange(AUDIO_NUM_SAMPLES, dtype=np.float64) / AUDIO_SAMPLE_RATE

# Filter designed at the correct rate — normalised cutoff is now 15000/24000 = 0.625
AUDIO_SOS = signal.butter(
    N=6,
    Wn=FM_AUDIO_MAX_HZ,         # 15 000 Hz
    btype="lowpass",
    fs=AUDIO_SAMPLE_RATE,       # 48 000 Hz  ← matches the domain
    output="sos",
)


# -------------------------- UTILS -------------------------- #
def ensure_output_dirs() -> None:
    for sub in ("iq", "metadata", "plots"):
        (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)


def mean_power(x: np.ndarray) -> float:
    """
    Mean linear power of a real or complex ndarray.
    """
    return float(np.mean(np.abs(x) ** 2))


def safe_ratio_db(numerator: float, denominator: float) -> float:
    """
    Convert a power ratio to dB with floor protection.
    """
    return float(10.0 * np.log10(max(numerator, 1e-12) / max(denominator, 1e-12)))


def quantize_complex(x: np.ndarray, bits: int = 8):
    """
    Quantize complex samples as if they passed through an 8-bit ADC.

    Returns
    -------
    xq : np.ndarray
        Quantized and renormalized complex64 samples.
    clip_fraction : float
        Fraction of samples whose I or Q exceeded [-1, 1] before quantization.
    """
    if not np.iscomplexobj(x):
        raise TypeError("x must be a complex-valued ndarray")

    qmax = (1 << (bits - 1)) - 1      # 127
    qmin = -(1 << (bits - 1))         # -128

    xr = x.real
    xi = x.imag

    clipped = (xr > 1.0) | (xr < -1.0) | (xi > 1.0) | (xi < -1.0)
    clip_fraction = float(np.mean(clipped))

    xr_c = np.clip(xr, -1.0, 1.0)
    xi_c = np.clip(xi, -1.0, 1.0)

    xr_q = np.clip(np.floor(xr_c * 128.0), qmin, qmax).astype(np.int8, copy=False)
    xi_q = np.clip(np.floor(xi_c * 128.0), qmin, qmax).astype(np.int8, copy=False)

    xq = (xr_q.astype(np.float32) + 1j * xi_q.astype(np.float32)) / 128.0
    return xq.astype(np.complex64, copy=False), clip_fraction


def apply_iq_imbalance(x: np.ndarray, gain_db: float, phase_deg: float) -> np.ndarray:
    """
    Widely-linear IQ imbalance model:
        y[n] = alpha * x[n] + beta * conj(x[n])
    """
    g = np.float32(10.0 ** (gain_db / 20.0))
    phi = np.float32(np.deg2rad(phase_deg))

    alpha = np.complex64(0.5 * (1.0 + g * np.exp(-1j * phi)))
    beta = np.complex64(0.5 * (1.0 - g * np.exp(+1j * phi)))

    y = alpha * x + beta * np.conj(x)
    return y.astype(np.complex64, copy=False)


def choose_station_offsets(rng: np.random.Generator, num_stations: int):
    """
    Choose station offsets on an FM-like raster while enforcing minimum separation.
    """
    max_offset = int(SAMPLE_RATE // 2 - EDGE_GUARD_HZ)
    candidates = np.arange(-max_offset, max_offset + 1, CHANNEL_SPACING_HZ, dtype=np.int64)
    rng.shuffle(candidates)

    selected = []
    for f in candidates:
        if all(abs(int(f) - int(s)) >= MIN_STATION_SEPARATION_HZ for s in selected):
            selected.append(int(f))
            if len(selected) == num_stations:
                break

    if len(selected) < num_stations:
        raise RuntimeError("Could not place all stations with current spacing constraints.")

    selected.sort()
    return selected


def synth_audio_message(rng: np.random.Generator):
    audio = np.zeros(AUDIO_NUM_SAMPLES, dtype=np.float32)

    n_tones = int(rng.integers(2, 6))
    tone_freqs, tone_amps = [], []

    for _ in range(n_tones):
        f  = float(rng.uniform(300.0, 15_000.0))
        a  = float(rng.uniform(0.15, 1.0))
        ph = float(rng.uniform(0.0, 2.0 * np.pi))
        audio += np.float32(a) * np.sin(2.0 * np.pi * f * T_AUDIO + ph).astype(np.float32)
        tone_freqs.append(f)
        tone_amps.append(a)

    white   = rng.standard_normal(AUDIO_NUM_SAMPLES, dtype=np.float32)
    colored = signal.sosfilt(AUDIO_SOS, white).astype(np.float32)   # well-conditioned now
    std = float(np.std(colored))
    if std > 1e-12:
        colored /= np.float32(std)

    noise_mix = float(rng.uniform(0.03, 0.20))
    audio += np.float32(noise_mix) * colored

    peak = float(np.max(np.abs(audio)))
    if peak > 1e-12:
        audio /= np.float32(peak)

    # Upsample from 48 kHz → 8 MHz using polyphase resampler
    audio_rf = signal.resample_poly(audio, _UP, _DOWN).astype(np.float32)

    # Guard against resampler overshoot before trimming
    peak_rf = float(np.max(np.abs(audio_rf)))
    if peak_rf > 1e-12:
        audio_rf /= np.float32(peak_rf)

    if len(audio_rf) >= NUM_IQ_SAMPLES:
        audio_rf = audio_rf[:NUM_IQ_SAMPLES]
    else:
        audio_rf = np.pad(audio_rf, (0, NUM_IQ_SAMPLES - len(audio_rf)))

    # Trim or pad to exactly NUM_IQ_SAMPLES (rounding may cause ±1 sample)
    if len(audio_rf) >= NUM_IQ_SAMPLES:
        audio_rf = audio_rf[:NUM_IQ_SAMPLES]
    else:
        audio_rf = np.pad(audio_rf, (0, NUM_IQ_SAMPLES - len(audio_rf)))

    meta = {
        "num_tones": n_tones,
        "tone_freqs_hz": [float(v) for v in tone_freqs],
        "tone_amps":     [float(v) for v in tone_amps],
        "noise_mix":     noise_mix,
    }
    return audio_rf, meta


def synth_fm_station(rng: np.random.Generator, freq_offset_hz: float, gain_db: float):
    """
    Create one complex WBFM station at the requested carrier offset.
    """
    audio, audio_meta = synth_audio_message(rng)
    deviation_hz = float(rng.uniform(FM_DEVIATION_HZ_MIN, FM_DEVIATION_HZ_MAX))
    init_phase = float(rng.uniform(0.0, 2.0 * np.pi))

    phase = np.empty(NUM_IQ_SAMPLES, dtype=np.float64)
    np.cumsum(audio, dtype=np.float64, out=phase)
    phase *= 2.0 * np.pi * (deviation_hz / SAMPLE_RATE)
    phase += 2.0 * np.pi * freq_offset_hz * T_VECTOR
    phase += init_phase

    station = np.exp(1j * phase).astype(np.complex64, copy=False)
    station *= np.float32(10.0 ** (gain_db / 20.0))

    meta = {
        "freq_offset_hz": float(freq_offset_hz),
        "gain_db": float(gain_db),
        "deviation_hz": deviation_hz,
        "audio": audio_meta,
    }
    return station, meta


def add_awgn(x: np.ndarray, rng: np.random.Generator, snr_db: float):
    """
    Add complex AWGN to achieve the requested SNR relative to current signal power.
    """
    sig_pow = mean_power(x)
    sig_pow = max(sig_pow, 1e-12)

    noise_pow = sig_pow / (10.0 ** (snr_db / 10.0))
    sigma = np.float32(np.sqrt(noise_pow / 2.0))

    nr = rng.standard_normal(x.shape[0], dtype=np.float32)
    ni = rng.standard_normal(x.shape[0], dtype=np.float32)
    noise = sigma * (nr + 1j * ni)
    noise = noise.astype(np.complex64, copy=False)

    y = x + noise
    return y.astype(np.complex64, copy=False), noise, float(noise_pow)


def plot_capture(iq: np.ndarray, metadata: dict, out_path: Path) -> None:
    """
    Save a PSD plot and mark nominal station centers.
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    try:
        f, pxx = signal.welch(
            iq,
            fs=SAMPLE_RATE,
            window="hann",
            nperseg=PSD_NPERSEG,
            noverlap=PSD_NPERSEG // 2,
            detrend=False,
            return_onesided=False,
            scaling="density",
        )

        f = np.fft.fftshift(f)
        pxx = np.fft.fftshift(pxx)

        ax.plot(f / 1e6, 10.0 * np.log10(pxx + 1e-18), linewidth=1.0)

        for st in metadata["stations"]:
            ax.axvline(st["freq_offset_hz"] / 1e6, linestyle="--", linewidth=0.8)

        ax.set_title(f"Synthetic HackRF-like FM Capture #{metadata['index']:03d}")
        ax.set_xlabel("Baseband frequency offset (MHz)")
        ax.set_ylabel("PSD (dB)")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(out_path, dpi=150)
    finally:
        plt.close(fig)

def measure_adc_utilization(x: np.ndarray) -> dict:
    """
    Measure how strongly the pre-quantization signal uses the ADC full-scale range.
    Full scale is defined per component over [-1, 1].
    """
    xr = x.real.astype(np.float32, copy=False)
    xi = x.imag.astype(np.float32, copy=False)

    abs_i = np.abs(xr)
    abs_q = np.abs(xi)
    abs_comp = np.maximum(abs_i, abs_q)

    rms_i = float(np.sqrt(np.mean(xr ** 2)))
    rms_q = float(np.sqrt(np.mean(xi ** 2)))
    rms_comp = float(np.sqrt(0.5 * (np.mean(xr ** 2) + np.mean(xi ** 2))))

    peak_i = float(np.max(abs_i))
    peak_q = float(np.max(abs_q))
    peak_comp = float(np.max(abs_comp))

    p99_comp = float(np.percentile(abs_comp, 99.0))
    p999_comp = float(np.percentile(abs_comp, 99.9))

    return {
        "adc_peak_component_fs": peak_comp,
        "adc_peak_i_fs": peak_i,
        "adc_peak_q_fs": peak_q,
        "adc_rms_component_fs": rms_comp,
        "adc_rms_i_fs": rms_i,
        "adc_rms_q_fs": rms_q,
        "adc_p99_component_fs": p99_comp,
        "adc_p999_component_fs": p999_comp,
        "adc_peak_component_dbfs": float(20.0 * np.log10(max(peak_comp, 1e-12))),
        "adc_rms_component_dbfs": float(20.0 * np.log10(max(rms_comp, 1e-12))),
        "adc_underutilized_peak_lt_0p25_fs": bool(peak_comp < 0.25),
        "adc_underutilized_rms_lt_0p10_fs": bool(rms_comp < 0.10),
    }
# -------------------------- CORE WORKER -------------------------- #
def generate_worker(idx: int):
    """
    Generate one synthetic wideband capture containing multiple FM stations,
    apply HackRF-like impairments, quantize, and save artifacts.
    """
    rng = np.random.default_rng(SEED_BASE + idx)

    num_stations = int(rng.integers(2, 5))
    station_offsets = choose_station_offsets(rng, num_stations)
    station_gain_dbs = [float(rng.uniform(*STATION_GAIN_DB_RANGE)) for _ in range(num_stations)]

    clean_capture = np.zeros(NUM_IQ_SAMPLES, dtype=np.complex64)
    stations_meta = []

    # 1) Build clean multi-station capture
    for f_off, g_db in zip(station_offsets, station_gain_dbs):
        station_iq, st_meta = synth_fm_station(rng, float(f_off), g_db)
        clean_capture += station_iq
        stations_meta.append(st_meta)

    # 2) Add AWGN
    snr_db = float(rng.uniform(*NOISE_SNR_DB_RANGE))
    capture_awgn, awgn_component, noise_power = add_awgn(clean_capture, rng, snr_db)

    # 3) Apply IQ imbalance
    iq_gain_db = float(rng.uniform(*IQ_GAIN_IMBALANCE_DB_RANGE))
    iq_phase_deg = float(rng.uniform(*IQ_PHASE_IMBALANCE_DEG_RANGE))
    signal_branch = apply_iq_imbalance(clean_capture, iq_gain_db, iq_phase_deg)
    noise_branch = apply_iq_imbalance(awgn_component, iq_gain_db, iq_phase_deg)
    capture = signal_branch + noise_branch

    # 4) Add DC offset in place
    dc_amp = float(rng.uniform(*DC_OFFSET_RANGE))
    dc_phase = float(rng.uniform(0.0, 2.0 * np.pi))
    dc = np.complex64(dc_amp * np.exp(1j * dc_phase))
    capture += dc

    # 5) ADC drive in place
    adc_drive = float(rng.uniform(*ADC_DRIVE_RANGE))
    drive_scale = np.float32(adc_drive)
    capture *= drive_scale
    signal_branch *= drive_scale
    noise_branch *= drive_scale
    dc_after_drive = np.complex64(dc * drive_scale)

    # 5a) Measure ADC utilization before quantization
    adc_utilization = measure_adc_utilization(capture)

    # 6) Quantize
    capture_q, clip_fraction = quantize_complex(capture, bits=8)

    # 6b) Post-impairment power bookkeeping
    signal_power_post = mean_power(signal_branch)
    awgn_power_post = mean_power(noise_branch)
    dc_power_post = float(np.abs(dc_after_drive) ** 2)

    quantization_clipping_error = capture_q.astype(np.complex64, copy=False) - capture.astype(np.complex64, copy=False)
    quantization_clipping_error_power = mean_power(quantization_clipping_error)

    post_awgn_snr_db = safe_ratio_db(signal_power_post, awgn_power_post)
    post_signal_to_disturbance_db = safe_ratio_db(
        signal_power_post,
        awgn_power_post + dc_power_post,
    )
    post_sindr_db = safe_ratio_db(
        signal_power_post,
        awgn_power_post + dc_power_post + quantization_clipping_error_power,
    )

    # 7) Save IQ
    iq_path = OUT_DIR / "iq" / f"sample_{idx:03d}.npy"
    np.save(iq_path, capture_q)

    # 8) Metadata
    plot_path = OUT_DIR / "plots" / f"sample_{idx:03d}.png"
    meta_path = OUT_DIR / "metadata" / f"sample_{idx:03d}.json"

    metadata = {
        "index": idx,
        "status": "success",
        "sample_rate_hz": SAMPLE_RATE,
        "num_iq_samples": NUM_IQ_SAMPLES,
        "center_freq_hz": CENTER_FREQ_HZ,
        "num_stations": num_stations,
        "stations": stations_meta,
        "awgn_injection_snr_db": snr_db,
        "awgn_injection_noise_power_linear": noise_power,
        "label_semantics": {
            "awgn_injection_snr_db": "SNR at the AWGN injection point, before IQ imbalance, DC offset, ADC drive, and quantization.",
            "awgn_injection_noise_power_linear": "AWGN power at the injection point, before downstream impairments.",
            "post_awgn_snr_db": "Signal-to-AWGN ratio after IQ imbalance and ADC drive, but excluding DC and quantization/clipping error.",
            "post_signal_to_disturbance_db": "Signal-to-(AWGN+DC) ratio after analog impairments and ADC drive.",
            "post_sindr_db": "Signal-to-(AWGN+DC+quantization/clipping error) ratio at the final ADC output.",
        },
        "post_impairment_metrics": {
            "signal_power_linear": signal_power_post,
            "awgn_power_linear": awgn_power_post,
            "dc_power_linear": dc_power_post,
            "quantization_clipping_error_power_linear": quantization_clipping_error_power,
            "adc_utilization_pre_quantization": adc_utilization,
            "post_awgn_snr_db": post_awgn_snr_db,
            "post_signal_to_disturbance_db": post_signal_to_disturbance_db,
            "post_sindr_db": post_sindr_db,
        },
        "impairments": {
            "dc_offset_amplitude": dc_amp,
            "dc_offset_phase_deg": float(np.rad2deg(dc_phase)),
            "iq_gain_imbalance_db": iq_gain_db,
            "iq_phase_imbalance_deg": iq_phase_deg,
            "adc_drive": adc_drive,
            "clip_fraction": clip_fraction,
        },
        "paths": {
            "iq_npy": str(iq_path),
            "plot_png": str(plot_path) if ENABLE_PLOTS else None,
            "metadata_json": str(meta_path),
        },
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    # 9) Optional plot
    if ENABLE_PLOTS:
        plot_capture(capture_q, metadata, plot_path)

    return {
        "index": idx,
        "status": "success",
        "num_stations": num_stations,
        "clip_fraction": clip_fraction,
        "post_sindr_db": post_sindr_db,
    }


# -------------------------- MAIN -------------------------- #
if __name__ == "__main__":
    ensure_output_dirs()

    print(f"Starting parallel generation of {NUM_EXAMPLES} synthetic FM captures...")

    max_workers = min(NUM_EXAMPLES, max(1, (os.cpu_count() or 1) // 2))

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(generate_worker, range(NUM_EXAMPLES)))

    print(f"Successfully generated dataset at: {OUT_DIR}")
    print("Results:")
    for r in results:
        print(r)

Starting parallel generation of 3 synthetic FM captures...
Successfully generated dataset at: dataset_fm_hackrf_3
Results:
{'index': 0, 'status': 'success', 'num_stations': 4, 'clip_fraction': 0.5142745971679688, 'post_sindr_db': 6.009617696790008}
{'index': 1, 'status': 'success', 'num_stations': 3, 'clip_fraction': 0.4552040100097656, 'post_sindr_db': 10.227885810897973}
{'index': 2, 'status': 'success', 'num_stations': 3, 'clip_fraction': 0.4317474365234375, 'post_sindr_db': 7.9674596699299824}


Output Artifacts
For each generated example:

- IQ Data: Saved as a .npy file (complex64, after quantization).

- Metadata: A JSON file containing:

Station details (frequency offset, gain, deviation, audio composition)

Impairment parameters (DC offset, IQ imbalance, ADC drive)

Power metrics (signal, noise, DC, quantization error)

SNR/SINDR calculations at various stages

ADC utilization statistics

- Optional Plot: PSD plot with station frequency markers (if ENABLE_PLOTS is set).



<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c4818a93-f2bf-4892-b7a2-47816cd4c47d' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>